# Sample 89 `6-seged.tiff` digital rock and pore-network workflow

This notebook runs in the Conda `ml` environment and reproduces the complete local workflow for:

1. Remapping `6-seged.tiff` to a binary segmented core (`solid=255`, `pore=0`).
2. Exporting a full-resolution Fiji/ImageJ-style interactive HTML volume.
3. Exporting a pnextract pore-network model and ball-stick HTML visualization.
4. Exporting a `visualize_digital_rock_3d.py` digital-rock preview as both PNG and interactive HTML.

The input file uses labels `1` and `2`; this notebook maps the majority label to solid (`255`) and the remaining label(s) to pore (`0`).

In [22]:
from pathlib import Path
import json
import os
import subprocess
import sys
import time

import numpy as np
import tifffile

PROJECT_ROOT = Path(r"C:\Users\imgw\Documents\Codex\SIP模拟\sip模拟")
os.chdir(PROJECT_ROOT)
PYTHON = Path(sys.executable)

INPUT_TIFF = r"C:\Users\imgw\Documents\Codex\SIP模拟\sip模拟\data_inventory\ct_backed_samples_raw_copy_20260605\sample_16_Grainstone\CT_slices\1-CTseg\5-16seged.tiff"
SAMPLE_ID = "5-16seged"

VOXEL_SIZE_UM = 1.92
NETWORK_DOWNSAMPLE = 1
VISUALIZE_DOWNSAMPLE = 4

# 当前案例的统一输出目录
CASE_OUTPUT_DIR = PROJECT_ROOT / "results" / "pore_network" / SAMPLE_ID

# 为了清晰，仍然分几个子文件夹，但都在 CASE_OUTPUT_DIR 下面
DERIVED_DIR = CASE_OUTPUT_DIR / "binary"
FIGURE_DIR = CASE_OUTPUT_DIR / "visualization"
SOURCE_DATA_DIR = CASE_OUTPUT_DIR / "metadata"
PN_INPUT_DIR = CASE_OUTPUT_DIR / "pnextract_inputs"
PN_NETWORK_DIR = CASE_OUTPUT_DIR / "pnextract_network"

for folder in [CASE_OUTPUT_DIR, DERIVED_DIR, FIGURE_DIR, SOURCE_DATA_DIR, PN_INPUT_DIR, PN_NETWORK_DIR]:
    folder.mkdir(parents=True, exist_ok=True)


BINARY_TIFF = DERIVED_DIR / f"{SAMPLE_ID}_solid255_pore0.tiff"
BINARY_RAW = DERIVED_DIR / f"{SAMPLE_ID}_solid255_pore0.raw"
REMAP_METADATA = SOURCE_DATA_DIR / f"{SAMPLE_ID}_solid255_pore0_remap_metadata.json"

FIJI_HTML = FIGURE_DIR / f"{SAMPLE_ID}_fiji3d_fullres_volume_interactive.html"
FIJI_METADATA = SOURCE_DATA_DIR / f"{SAMPLE_ID}_fiji3d_fullres_volume_interactive_metadata.json"

PN_HTML = FIGURE_DIR / f"{SAMPLE_ID}_pnextract_ballstick_interactive.html"
PN_METADATA = SOURCE_DATA_DIR / f"{SAMPLE_ID}_pnextract_ballstick_interactive_metadata.json"

DIGITAL_ROCK_PNG = FIGURE_DIR / f"{SAMPLE_ID}_visualize_digital_rock_pore_preview.png"
DIGITAL_ROCK_HTML = FIGURE_DIR / f"{SAMPLE_ID}_visualize_digital_rock_pore_preview_interactive.html"
DIGITAL_ROCK_METADATA = SOURCE_DATA_DIR / f"{SAMPLE_ID}_visualize_digital_rock_pore_preview_metadata.json"

print("Python:", PYTHON)
print("Project root:", PROJECT_ROOT)
# print("Input TIFF exists:", INPUT_TIFF.exists(), INPUT_TIFF)

Python: c:\Users\imgw\.conda\envs\ml\python.exe
Project root: C:\Users\imgw\Documents\Codex\SIP模拟\sip模拟


In [23]:
def run_command(command, *, cwd=PROJECT_ROOT, timeout=None):
    print("\n$", " ".join(str(part) for part in command))
    t0 = time.time()
    result = subprocess.run(
        [str(part) for part in command],
        cwd=str(cwd),
        text=True,
        encoding="utf-8",
        errors="replace",
        capture_output=True,
        timeout=timeout,
    )
    elapsed = time.time() - t0
    print(f"elapsed: {elapsed:.1f} s")
    if result.stdout:
        print("stdout:\n", result.stdout[-4000:])
    if result.stderr:
        print("stderr:\n", result.stderr[-4000:])
    if result.returncode != 0:
        raise RuntimeError(f"command failed with exit code {result.returncode}")
    return result


def file_report(path: Path):
    path = Path(path)
    return {
        "path": str(path),
        "exists": path.exists(),
        "size_mb": round(path.stat().st_size / 1024 / 1024, 3) if path.exists() else None,
    }

## 1. Remap labels to `solid=255`, `pore=0`

For this file, label `1` is the majority component and is treated as solid. Label `2` is treated as pore. The rule below still computes this from the actual voxel counts so the notebook records the evidence.

In [24]:
volume = tifffile.imread(INPUT_TIFF)
if volume.ndim != 3:
    raise ValueError(f"Expected a 3-D TIFF stack, got shape {volume.shape}")

values, counts = np.unique(volume, return_counts=True)
value_counts = {int(v): int(c) for v, c in zip(values, counts)}
solid_source_values = [int(values[int(np.argmax(counts))])]
pore_source_values = [int(v) for v in values if int(v) not in solid_source_values]

binary = np.where(np.isin(volume, solid_source_values), 255, 0).astype(np.uint8)
pore_voxels = int(np.count_nonzero(binary == 0))
solid_voxels = int(np.count_nonzero(binary == 255))
total_voxels = int(binary.size)
porosity = pore_voxels / total_voxels

# Preserve z-y-x voxel order from tifffile for both TIFF and RAW outputs.
tifffile.imwrite(BINARY_TIFF, binary, photometric="minisblack")
binary.tofile(BINARY_RAW)

remap_metadata = {
    "input_tiff": str(INPUT_TIFF),
    "output_tiff": str(BINARY_TIFF),
    "output_raw": str(BINARY_RAW),
    "shape_zyx": [int(v) for v in binary.shape],
    "dtype": str(binary.dtype),
    "source_value_counts": value_counts,
    "solid_source_values": solid_source_values,
    "pore_source_values": pore_source_values,
    "solid_output_value": 255,
    "pore_output_value": 0,
    "solid_voxels": solid_voxels,
    "pore_voxels": pore_voxels,
    "total_voxels": total_voxels,
    "porosity": porosity,
    "voxel_size_um_assumed_xyz": [VOXEL_SIZE_UM, VOXEL_SIZE_UM, VOXEL_SIZE_UM],
}
REMAP_METADATA.write_text(json.dumps(remap_metadata, indent=2, ensure_ascii=False), encoding="utf-8")

print(json.dumps(remap_metadata, indent=2, ensure_ascii=False))

{
  "input_tiff": "C:\\Users\\imgw\\Documents\\Codex\\SIP模拟\\sip模拟\\data_inventory\\ct_backed_samples_raw_copy_20260605\\sample_16_Grainstone\\CT_slices\\1-CTseg\\5-16seged.tiff",
  "output_tiff": "C:\\Users\\imgw\\Documents\\Codex\\SIP模拟\\sip模拟\\results\\pore_network\\5-16seged\\binary\\5-16seged_solid255_pore0.tiff",
  "output_raw": "C:\\Users\\imgw\\Documents\\Codex\\SIP模拟\\sip模拟\\results\\pore_network\\5-16seged\\binary\\5-16seged_solid255_pore0.raw",
  "shape_zyx": [
    800,
    666,
    666
  ],
  "dtype": "uint8",
  "source_value_counts": {
    "1": 287048552,
    "2": 67796248
  },
  "solid_source_values": [
    1
  ],
  "pore_source_values": [
    2
  ],
  "solid_output_value": 255,
  "pore_output_value": 0,
  "solid_voxels": 287048552,
  "pore_voxels": 67796248,
  "total_voxels": 354844800,
  "porosity": 0.19105887418950482,
  "voxel_size_um_assumed_xyz": [
    1.92,
    1.92,
    1.92
  ]
}


## 2. Export full-resolution Fiji/ImageJ-style interactive HTML

This uses the real VTK volume actor route from `render_segmented_core_fiji3d_html.py`. It keeps the full `300 x 300 x 300` volume (`--downsample 1`) and opens with the pore phase visible.

In [25]:
FIJI_CMD = [
    PYTHON,
    PROJECT_ROOT / "code" / "scripts" / "digital_rock_visualization" / "render_segmented_core_fiji3d_html.py",
    "--input", BINARY_TIFF,
    "--out", FIJI_HTML,
    "--metadata-out", FIJI_METADATA,
    "--solid-value", "255",
    "--downsample", "1",
    "--voxel-size-um", str(VOXEL_SIZE_UM),
    "--components", "0", "255",
    "--initial-visible", "pore",
    "--interpolation", "linear",
]
run_command(FIJI_CMD)
print(file_report(FIJI_HTML))
print(file_report(FIJI_METADATA))


$ c:\Users\imgw\.conda\envs\ml\python.exe C:\Users\imgw\Documents\Codex\SIP模拟\sip模拟\code\scripts\digital_rock_visualization\render_segmented_core_fiji3d_html.py --input C:\Users\imgw\Documents\Codex\SIP模拟\sip模拟\results\pore_network\5-16seged\binary\5-16seged_solid255_pore0.tiff --out C:\Users\imgw\Documents\Codex\SIP模拟\sip模拟\results\pore_network\5-16seged\visualization\5-16seged_fiji3d_fullres_volume_interactive.html --metadata-out C:\Users\imgw\Documents\Codex\SIP模拟\sip模拟\results\pore_network\5-16seged\metadata\5-16seged_fiji3d_fullres_volume_interactive_metadata.json --solid-value 255 --downsample 1 --voxel-size-um 1.92 --components 0 255 --initial-visible pore --interpolation linear
elapsed: 24.7 s
stdout:
 {
  "input_tiff": "C:\\Users\\imgw\\Documents\\Codex\\SIP模拟\\sip模拟\\results\\pore_network\\5-16seged\\binary\\5-16seged_solid255_pore0.tiff",
  "output_html": "C:\\Users\\imgw\\Documents\\Codex\\SIP模拟\\sip模拟\\results\\pore_network\\5-16seged\\visualization\\5-16seged_fiji3d_full

## 3. Export pnextract pore network and ball-stick HTML

The pnextract step uses the binary TIFF, maps `0` to pore/void and non-pore to solid, and uses `downsample=4` by default so the network extraction and HTML remain practical on this workstation. Change `NETWORK_DOWNSAMPLE` in the first cell if you need another resolution.

In [26]:
PN_CMD = [
    PYTHON,
    PROJECT_ROOT / "code" / "scripts" / "pore_network" / "run_segmented_core_pnextract_ballstick.py",
    "--input", BINARY_TIFF,
    "--title", f"{SAMPLE_ID}_pnextract",
    "--pore-values", "0",
    "--solid-value", "255",
    "--voxel-size-um", str(VOXEL_SIZE_UM),
    "--downsample", str(NETWORK_DOWNSAMPLE),
    "--prepare-dir", PN_INPUT_DIR,
    "--network-dir", PN_NETWORK_DIR,
    "--html-out", PN_HTML,
    "--metadata-out", PN_METADATA,
]
run_command(PN_CMD)
print(file_report(PN_HTML))
print(file_report(PN_METADATA))
print("Parsed network dir:", PN_NETWORK_DIR / "network_parsed")


$ c:\Users\imgw\.conda\envs\ml\python.exe C:\Users\imgw\Documents\Codex\SIP模拟\sip模拟\code\scripts\pore_network\run_segmented_core_pnextract_ballstick.py --input C:\Users\imgw\Documents\Codex\SIP模拟\sip模拟\results\pore_network\5-16seged\binary\5-16seged_solid255_pore0.tiff --title 5-16seged_pnextract --pore-values 0 --solid-value 255 --voxel-size-um 1.92 --downsample 1 --prepare-dir C:\Users\imgw\Documents\Codex\SIP模拟\sip模拟\results\pore_network\5-16seged\pnextract_inputs --network-dir C:\Users\imgw\Documents\Codex\SIP模拟\sip模拟\results\pore_network\5-16seged\pnextract_network --html-out C:\Users\imgw\Documents\Codex\SIP模拟\sip模拟\results\pore_network\5-16seged\visualization\5-16seged_pnextract_ballstick_interactive.html --metadata-out C:\Users\imgw\Documents\Codex\SIP模拟\sip模拟\results\pore_network\5-16seged\metadata\5-16seged_pnextract_ballstick_interactive_metadata.json
elapsed: 371.4 s
stdout:
 view_pores_points.vtp\",\n    \"throats_vtp\": \"C:\\\\Users\\\\imgw\\\\Documents\\\\Codex\\\\SIP模

## 4. Export `visualize_digital_rock_3d.py` preview as PNG and interactive HTML

This script reads RAW input, so the notebook also wrote a binary RAW copy in step 1. The preview is downsampled for interactive mesh size; the full-resolution interactive volume is already exported in step 2.

In [27]:
shape_args = [str(v) for v in binary.shape]
VIS_CMD = [
    PYTHON,
    PROJECT_ROOT / "code" / "scripts" / "digital_rock_visualization" / "visualize_digital_rock_3d.py",
    "--raw", BINARY_RAW,
    "--shape", *shape_args,
    "--dtype", "uint8",
    "--pore-label", "0",
    "--solid-label", "255",
    "--crop-start", "0", "0", "0",
    "--crop-size", *shape_args,
    "--downsample", str(VISUALIZE_DOWNSAMPLE),
    "--phase", "pore",
    "--out", DIGITAL_ROCK_PNG,
    # "--html-out", DIGITAL_ROCK_HTML,
    "--metadata-out", DIGITAL_ROCK_METADATA,
]
run_command(VIS_CMD)
print(file_report(DIGITAL_ROCK_PNG))
print(file_report(DIGITAL_ROCK_HTML))
print(file_report(DIGITAL_ROCK_METADATA))


$ c:\Users\imgw\.conda\envs\ml\python.exe C:\Users\imgw\Documents\Codex\SIP模拟\sip模拟\code\scripts\digital_rock_visualization\visualize_digital_rock_3d.py --raw C:\Users\imgw\Documents\Codex\SIP模拟\sip模拟\results\pore_network\5-16seged\binary\5-16seged_solid255_pore0.raw --shape 800 666 666 --dtype uint8 --pore-label 0 --solid-label 255 --crop-start 0 0 0 --crop-size 800 666 666 --downsample 4 --phase pore --out C:\Users\imgw\Documents\Codex\SIP模拟\sip模拟\results\pore_network\5-16seged\visualization\5-16seged_visualize_digital_rock_pore_preview.png --metadata-out C:\Users\imgw\Documents\Codex\SIP模拟\sip模拟\results\pore_network\5-16seged\metadata\5-16seged_visualize_digital_rock_pore_preview_metadata.json
elapsed: 215.5 s
stdout:
 {
  "raw": "C:\\Users\\imgw\\Documents\\Codex\\SIP模拟\\sip模拟\\results\\pore_network\\5-16seged\\binary\\5-16seged_solid255_pore0.raw",
  "shape": [
    800,
    666,
    666
  ],
  "dtype": "uint8",
  "phase": "pore",
  "label": 0,
  "crop_start": [
    0,
    0,
    

## 5. Output checklist

In [29]:
outputs = {
    "binary_tiff": file_report(BINARY_TIFF),
    "binary_raw": file_report(BINARY_RAW),
    "remap_metadata": file_report(REMAP_METADATA),
    "fiji_fullres_html": file_report(FIJI_HTML),
    "fiji_metadata": file_report(FIJI_METADATA),
    "pnextract_ballstick_html": file_report(PN_HTML),
    "pnextract_metadata": file_report(PN_METADATA),
    "digital_rock_preview_png": file_report(DIGITAL_ROCK_PNG),
    # "digital_rock_preview_html": file_report(DIGITAL_ROCK_HTML),
    "digital_rock_metadata": file_report(DIGITAL_ROCK_METADATA),
}
print(json.dumps(outputs, indent=2, ensure_ascii=False))
missing = [name for name, report in outputs.items() if not report["exists"]]
if missing:
    raise RuntimeError(f"Missing outputs: {missing}")
print(f"Porosity from remapped voxels: {porosity:.6f} ({porosity * 100:.2f}%)")

{
  "binary_tiff": {
    "path": "C:\\Users\\imgw\\Documents\\Codex\\SIP模拟\\sip模拟\\results\\pore_network\\5-16seged\\binary\\5-16seged_solid255_pore0.tiff",
    "exists": true,
    "size_mb": 338.533
  },
  "binary_raw": {
    "path": "C:\\Users\\imgw\\Documents\\Codex\\SIP模拟\\sip模拟\\results\\pore_network\\5-16seged\\binary\\5-16seged_solid255_pore0.raw",
    "exists": true,
    "size_mb": 338.406
  },
  "remap_metadata": {
    "path": "C:\\Users\\imgw\\Documents\\Codex\\SIP模拟\\sip模拟\\results\\pore_network\\5-16seged\\metadata\\5-16seged_solid255_pore0_remap_metadata.json",
    "exists": true,
    "size_mb": 0.001
  },
  "fiji_fullres_html": {
    "path": "C:\\Users\\imgw\\Documents\\Codex\\SIP模拟\\sip模拟\\results\\pore_network\\5-16seged\\visualization\\5-16seged_fiji3d_fullres_volume_interactive.html",
    "exists": true,
    "size_mb": 14.993
  },
  "fiji_metadata": {
    "path": "C:\\Users\\imgw\\Documents\\Codex\\SIP模拟\\sip模拟\\results\\pore_network\\5-16seged\\metadata\\5-16seged_fi